In [ ]:
# 이 코드만 사용하시면 됩니다.. 데이터 새로 다운 받아서

In [ ]:
"""
공동주택 좌표 연결 + 증분 지오코딩 + 좌표정보(레지스트리) 갱신
──────────────────────────────────────────────────────────────
[흐름]
1) 새로 받은 데이터(NEW_FILE) 로드 + 도로명주소 추출
2) 아파트_좌표정보.csv(COORD_FILE)를 [아파트명, 아파트코드]로 병합 → 좌표 연결
3) 좌표가 비어 있는 '단지'만 SGIS 지오코딩 (행이 아니라 단지 단위로 1회)
4) 전체 결과 저장(OUT_FILE)
5) 아파트_좌표정보.csv 재생성(그룹화·세대수 합산) → 신규 단지 누적, 기존분 보존

[핵심] 매칭키 = 아파트코드(+아파트명). 도로명주소 표기 차이에 영향받지 않음.
       API 호출은 '좌표정보에 없는 신규 단지' 수만큼만 발생.
"""

import os
import pandas as pd
import requests
import time
from pyproj import Transformer

# ── 경로 ───────────────────────────────────────────────────────
NEW_FILE   = r"D:\Project\★아파트지도\data\공동주택현황.csv"            # 새로 받은 원본
COORD_FILE = r"D:\Project\★아파트지도\data\아파트_좌표정보.csv"          # 좌표 레지스트리(읽기+갱신)
OUT_FILE   = r"D:\Project\★아파트지도\data\공동주택현황_지오코딩완료.csv"  # 좌표 연결된 전체(행 단위)
PROGRESS   = r"D:\Project\★아파트지도\data\_좌표추출_진행중.csv"          # 지오코딩 중간저장

KEY      = ['아파트명', '아파트코드']                  # 매칭 기준
REG_COLS = ['아파트명', '아파트코드', '세대수', '도로명주소', '위도', '경도']  # 레지스트리 스키마

SERVICE_ID = "02db299fb553470b97aa"
SECRET_KEY = "381beff8ef174fee8691"

DELAY       = 0.15
MAX_RETRY   = 3
RETRY_DELAY = 2.0
SAVE_EVERY  = 50

# ── 좌표 변환기 (EPSG:5179 → WGS84) ─────────────────────────────
transformer = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)

def tm_to_wgs84(x, y):
    lon, lat = transformer.transform(x, y)
    return round(lat, 7), round(lon, 7)

def load_csv(path):
    last = None
    for enc in ['utf-8-sig', 'cp949', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last = e
    raise RuntimeError(f"읽기 실패: {path} ({last})")

# ── 1. 새 데이터 로드 + 도로명주소 추출 ──────────────────────────
print("=" * 60)
df = load_csv(NEW_FILE)
df['도로명주소'] = df['주소'].str.split('(').str[0].str.strip()

need = ['아파트명', '아파트코드', '세대수', '주소']
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"새 데이터에 필요한 컬럼 누락: {missing} | 현재 컬럼: {df.columns.tolist()}")
print(f"📂 새 데이터: {len(df):,}행 | 컬럼: {df.columns.tolist()}")

# ── 2. 좌표 레지스트리 병합 ──────────────────────────────────────
if os.path.exists(COORD_FILE):
    reg = load_csv(COORD_FILE)
    reg['세대수'] = pd.to_numeric(reg.get('세대수'), errors='coerce').fillna(0).astype(int)
    lookup = reg[KEY + ['위도', '경도']].drop_duplicates(subset=KEY)
    df = df.merge(lookup, on=KEY, how='left')
    print(f"📍 좌표 레지스트리: {len(reg):,}건 → 병합 완료")
else:
    reg = pd.DataFrame(columns=REG_COLS)
    df['위도'] = pd.NA
    df['경도'] = pd.NA
    print("📍 좌표 레지스트리 없음(최초 실행) → 전체 지오코딩 진행")

matched = df['위도'].notna().sum()
print(f"🔗 좌표 매칭: {matched:,}행 | 미매칭: {df['위도'].isna().sum():,}행")

# ── SGIS 토큰 / 지오코딩 ────────────────────────────────────────
def get_access_token():
    url = "https://sgisapi.mods.go.kr/OpenAPI3/auth/authentication.json"
    params = {"consumer_key": SERVICE_ID, "consumer_secret": SECRET_KEY}
    data = requests.get(url, params=params, timeout=10).json()
    if data.get("errCd") != 0:
        raise RuntimeError(f"Token 발급 실패: {data}")
    return data["result"]["accessToken"]

def geocode(address, token):
    url = "https://sgisapi.mods.go.kr/OpenAPI3/addr/geocode.json"
    params = {"accessToken": token, "address": address, "pagenum": 0, "resultcount": 1}
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    if data.get("errCd") != 0:
        return None
    rd = data.get("result", {}).get("resultdata", [])
    if not rd:
        return None
    r = rd[0]
    lat, lon = tm_to_wgs84(float(r.get("x", 0)), float(r.get("y", 0)))
    return {"위도": lat, "경도": lon}

# ── 3. 미매칭 '단지'만 지오코딩 ─────────────────────────────────
todo = (df[df['위도'].isna()][KEY + ['도로명주소']]
        .drop_duplicates(subset=KEY)
        .reset_index(drop=True))

coord_map = {}   # (아파트명, 아파트코드) -> {"위도":, "경도":}
if len(todo) == 0:
    print("\n✅ 신규 단지 없음 — 지오코딩 불필요")
else:
    print(f"\n{'='*60}\n🌐 신규 단지 {len(todo):,}곳 지오코딩 시작\n{'='*60}")
    token = get_access_token()
    fail, start = 0, time.time()
    rows_progress = []

    for n, row in todo.iterrows():
        key  = (row['아파트명'], row['아파트코드'])
        addr = row['도로명주소']

        result = None
        for attempt in range(1, MAX_RETRY + 1):
            try:
                result = geocode(addr, token)
                break
            except requests.exceptions.HTTPError as e:
                if e.response is not None and e.response.status_code == 401:
                    print("   ⚠️  Token 만료 → 재발급")
                    token = get_access_token()
                else:
                    time.sleep(RETRY_DELAY)
            except Exception:
                time.sleep(RETRY_DELAY)

        coord_map[key] = result
        if result:
            rows_progress.append({**dict(zip(KEY, key)), **result})
        else:
            fail += 1
            print(f"   ❌ 실패: {row['아파트명']} | {addr}")

        if (n + 1) % SAVE_EVERY == 0 or (n + 1) == len(todo):
            elapsed = time.time() - start
            remain  = (elapsed / (n + 1)) * (len(todo) - (n + 1))
            print(f"   📊 [{n+1:>5}/{len(todo):,}] 실패 {fail:,} | "
                  f"경과 {elapsed/60:.1f}분 | 잔여 {remain/60:.1f}분")
            if rows_progress:
                pd.DataFrame(rows_progress).to_csv(PROGRESS, index=False, encoding='utf-8-sig')

        time.sleep(DELAY)

    # ── 지오코딩 결과를 전체 데이터에 반영 ──────────────────────
    mask = df['위도'].isna()
    for idx in df[mask].index:
        g = coord_map.get((df.at[idx, '아파트명'], df.at[idx, '아파트코드']))
        if g:
            df.at[idx, '위도'] = g['위도']
            df.at[idx, '경도'] = g['경도']

# ── 4. 좌표 연결된 전체 데이터 저장 (업종분류/업종 등 새 컬럼 포함) ──
df.to_csv(OUT_FILE, index=False, encoding='utf-8-sig')
print(f"\n💾 전체 저장 → {OUT_FILE}")
print(f"   좌표 있음 {df['위도'].notna().sum():,} | 좌표 없음 {df['위도'].isna().sum():,}")

# ── 5. 아파트_좌표정보.csv 재생성 (그룹화 + 누적) ────────────────
src = df[df['위도'].notna()].copy()
src['세대수'] = pd.to_numeric(src['세대수'], errors='coerce').fillna(0).astype(int)

new_reg = (src.groupby(['아파트명', '아파트코드'], as_index=False)
              .agg({'세대수': 'sum', '도로명주소': 'first', '위도': 'first', '경도': 'first'}))
new_reg = new_reg[REG_COLS]

# 신규/갱신분 우선 + 이번 데이터에 없는 기존 등록분 보존
if len(reg):
    final_reg = (pd.concat([new_reg, reg[REG_COLS]], ignore_index=True)
                   .drop_duplicates(subset=KEY, keep='first'))
else:
    final_reg = new_reg

final_reg.to_csv(COORD_FILE, index=False, encoding='utf-8-sig')

print(f"\n{'='*60}")
print(f"✅ 좌표정보 갱신 완료 → {COORD_FILE}")
print(f"   기존 {len(reg):,} → 최종 {len(final_reg):,}건 "
      f"(신규 {len(final_reg) - len(reg):,})")
print(f"\n[미리보기 상위 5행]")
print(final_reg.head().to_string(index=False))